In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import pandas as pd
import time, os
from dotenv import load_dotenv
# from langchain.schema import HumanMessage, SystemMessage, AIMessage
from typing import Callable, Dict, List, Optional, Tuple, Union
from genai.schema import TextGenerationParameters, TextGenerationReturnOptions
from genai.exceptions import ApiNetworkException, ApiResponseException, ValidationError
from genai import Client, Credentials
import numpy as np
from genai.schema import (
    DecodingMethod,
    ModerationHAP,
    ModerationParameters,
)
import json

/Users/christodoulosconstantinides/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import logging
logging.basicConfig(level=logging.DEBUG)

In [3]:
model_id = 'meta-llama/llama-2-70b-chat'

In [4]:
load_dotenv()
api_key = os.getenv("GENAI_KEY", None) 
api_url = os.getenv("GENAI_API", None)
creds = Credentials(api_key, api_endpoint=api_url)
params = TextGenerationParameters(
    decoding_method="greedy",
    max_new_tokens=2000,
    min_new_tokens=200,
    temperature=0.5,
    top_k=50,
    top_p=1,
    stop_sequences=["(TOKENSTOP)"],
    return_options=TextGenerationReturnOptions(input_text=False, input_tokens=True)
)

In [6]:
client = Client(credentials=Credentials.from_env())

In [7]:
asset_name = 'Standby Generator'

In [8]:
def generate(prompt):
    res = client.text.generation.create(
        model_id=model_id,
        inputs=prompt,
        # set to ordered to True if you need results in the same order as prompts
        parameters=params
    )
    result = next(res)
    result = result.results[0].generated_text
    return result

In [9]:
SMESystemPrompt = """
You act as a reliability engineer who is expert in failure modes and effect analysis (FMEA) of asset 
reliability. You task is to provide a accurate information about asset's component, subcomponent, failure mode and
failure reason.
"""

JSONPrompt = '''
You will given a list in the same format as the following:
Some text before
1. Mammals:
* Human
* Monkey
* Dolphin
* Dogs
2. Reptils:
* Snake
* Crocodile
* Lizard
This is extra
The expected output is in JSON format as the following:
{
    "Mammals": ["Human", "Monkey", "Dolphin", "Dogs"],
    "Reptils": ["Snake", "Crocodile", "Lizard"]
}
Convert to a JSON format the following:
'''

In [10]:
# SensorExtractionPrompt = """
# I am monitoring the `Standby Generator` asset using IOT/OT sensor data.

# The asset `Standby Generator` can have multiple failure modes such as: `Engine failure, Alternator Failure, ...`

# `Engine` is one component of the asset `standby generator` and `Engine` component can have different failure events, such as: 
# `Failure of the engine to start, Engine overheating, ...`. 

# A failure event can cause the asset `Standby Generator` to stop working or decrease its operating performance. 

# The user will give the name of a failure event and you need to provide 
# which sensors are useful to detect the failure event along with their temporal behavior with respect to normal operating condition. 

# During the `Failure of the engine to start` associated with the `Engine` component of the `Standby Generator` failure event 
# the sensor readings exhibit a different behavior than during the normal operation of the asset `Standby Generator`

# `Failure of the engine to start` associated with the `Engine` component of the `Standby Generator` asset?
# """
# # If user gives list of questions, 
# #  then summary should be written based on questions content for a given asset class. 

In [11]:
SensorPrompt = """
You act as a reliability engineer who is expert in failure modes and effect analysis (FMEA) of asset 
reliability. You task is to help Data Science team who is developing an anomaly detection algorithm to monitor the <asset_name> asset using IOT/OT sensor data for early detection of failure/fault event. 
A failure event can cause the asset <asset_name> to stop working or decrease its operating performance. 
The asset <asset_name> can have multiple failure modes such as: <asset_failure_1>, <asset_failure_2>. 
Engine failure is associated with <component> component of the asset <asset_name>. <component> component can have different failure events, such as: 
<component_failure_1>, <component_failure_2>, .... 

The user will give the name of a failure event and you need to provide which sensors are useful to detect the failure event 
along with their temporal behavior with respect to normal operating condition. 
During the failure event, certain sensor readings exhibit a different behavior 
when compared to the normal operation of the asset <asset_name>. 
You will provide an accurate information about sensors that are associated with given failure event and its temporal behaviours. 
The temporal behaviors of failure events should be detected by the anomaly detection algorithm 
using time series sensor data and should be in a form of rule or condition and use work such as increase, decrease, etc. 
Failure event can be detected using single or combination of multiple sensors. 
In case of multiple sensors, temporal behavior should also include the extepcted correlation across sensors. 
If sensors are not suitable for detecting given failure, return I do not know. Do not generate any additional information.  

The failure event is <failure_event>. Which sensors are relevant? What is the temporal behavior of the sensors?
"""

In [27]:
SensorPrompt = """
You act as a reliability engineer who is expert in failure modes and effect analysis (FMEA) of asset 
reliability. You task is to help Data Science team who is developing an anomaly detection algorithm to monitor the <asset_name> asset using IOT/OT sensor data for early detection of failure/fault event. 
A failure event can cause the asset <asset_name> to stop working or decrease its operating performance. 

The failure event is <failure_event> in <component>. Which sensors are relevant? What is the temporal behavior of the sensors?

You are required to provide a list of sensors that are relevant to detect the failure event of <failure_event>. Additionally, you need to describe the temporal behavior of these sensors.

Note: The sensors should be from the IOT/OT sensor data.

Answer:

The following sensors are relevant to detect the failure event of <failure_event>:
"""

In [12]:
failure_modes_question = """What are different failure modes for each of the components of a standby generator? 
Provide them as a list
"""

In [13]:
sme_prompt = f'{SMESystemPrompt}\n{failure_modes_question}'
response_sme = generate(sme_prompt)

In [15]:
print(sme_prompt)


You act as a reliability engineer who is expert in failure modes and effect analysis (FMEA) of asset 
reliability. You task is to provide a accurate information about asset's component, subcomponent, failure mode and
failure reason.

What are different failure modes for each of the components of a standby generator? 
Provide them as a list



In [16]:
print(response_sme)
# parse the output and create multiple prompts and ask which sensor variable is related with the event 


Answer:

1. Engine:
* Fuel starvation due to low fuel level or clogged fuel filter
* Oil starvation due to low oil level or clogged oil filter
* Overheating due to coolant loss or malfunctioning cooling system
* Mechanical failure due to worn or damaged engine components
* Electrical failure due to faulty wiring or electrical components
2. Alternator:
* Overheating due to excessive load or malfunctioning cooling system
* Electrical failure due to faulty wiring or electrical components
* Mechanical failure due to worn or damaged alternator components
* Brush failure due to worn or damaged brushes
3. Transfer Switch:
* Electrical failure due to faulty wiring or electrical components
* Mechanical failure due to worn or damaged transfer switch components
* Failure to transfer due to stuck or damaged switch
4. Battery:
* Failure due to low battery charge or aged batteries
* Electrical failure due to faulty wiring or electrical components
* Mechanical failure due to worn or damaged battery 

In [17]:
json_prompt = f'{JSONPrompt}\n{response_sme}'
response_json = generate(json_prompt)

In [18]:
start = response_json.find('{')
end = response_json.rfind('}')
failure_modes_json = json.loads(response_json[start:end+1])

In [19]:
failure_modes_json

{'Engine': ['Fuel starvation due to low fuel level or clogged fuel filter',
  'Oil starvation due to low oil level or clogged oil filter',
  'Overheating due to coolant loss or malfunctioning cooling system',
  'Mechanical failure due to worn or damaged engine components',
  'Electrical failure due to faulty wiring or electrical components'],
 'Alternator': ['Overheating due to excessive load or malfunctioning cooling system',
  'Electrical failure due to faulty wiring or electrical components',
  'Mechanical failure due to worn or damaged alternator components',
  'Brush failure due to worn or damaged brushes'],
 'Transfer Switch': ['Electrical failure due to faulty wiring or electrical components',
  'Mechanical failure due to worn or damaged transfer switch components',
  'Failure to transfer due to stuck or damaged switch'],
 'Battery': ['Failure due to low battery charge or aged batteries',
  'Electrical failure due to faulty wiring or electrical components',
  'Mechanical failure

In [26]:
print(modified_sensor_prompt)


You act as a reliability engineer who is expert in failure modes and effect analysis (FMEA) of asset 
reliability. You task is to help Data Science team who is developing an anomaly detection algorithm to monitor the Standby Generator asset using IOT/OT sensor data for early detection of failure/fault event. 
A failure event can cause the asset Standby Generator to stop working or decrease its operating performance. 
The asset Standby Generator can have multiple failure modes such as: Engine, Alternator. 
Engine failure is associated with Engine component of the asset Standby Generator. Engine component can have different failure events, such as: 
Fuel starvation due to low fuel level or clogged fuel filter, Oil starvation due to low oil level or clogged oil filter, .... 

The user will give the name of a failure event and you need to provide which sensors are useful to detect the failure event 
along with their temporal behavior with respect to normal operating condition. 
During the

In [28]:
results = {}
all_components = list(failure_modes_json.keys())
for component in failure_modes_json:
    results[component] = []
    for failure_mode in failure_modes_json[component]:
        replace_str = {
            "asset_name": asset_name,
            "component_failure_1": failure_modes_json[component][0],
            "component_failure_2": failure_modes_json[component][1] if len(failure_modes_json[component]) > 1 else 'etc',
            "component": component,
            "asset_failure_1": all_components[0],
            "asset_failure_2": all_components[1] if len(all_components) > 1 else 'etc',
            "failure_event": failure_mode
        }
        modified_sensor_prompt = SensorPrompt
        for key in replace_str:
            modified_sensor_prompt = modified_sensor_prompt.replace(f'<{key}>', replace_str[key])
        # print(modified_sensor_prompt)
        logging.debug(f'{asset_name}, {component}, {failure_mode}')
        res = generate(modified_sensor_prompt)
        results[component].append(res)
        logging.debug(res)
        with open(f'{asset_name}.json', 'w') as f:
            f.write(json.dumps(results))

Standby Generator, Engine, Fuel starvation due to low fuel level or clogged fuel filter

1. Fuel level sensor: This sensor measures the fuel level in the tank and can detect low fuel levels that can lead to fuel starvation. The temporal behavior of this sensor is that it will show a steady decrease in fuel level over time as the generator consumes fuel.
2. Fuel flow rate sensor: This sensor measures the rate at which fuel is being consumed by the generator. A decrease in fuel flow rate can indicate a clogged fuel filter or low fuel level. The temporal behavior of this sensor is that it will show a decrease in fuel flow rate over time as the filter becomes clogged or fuel level decreases.
3. Engine temperature sensor: A clogged fuel filter or low fuel level can cause the engine temperature to increase, which can lead to engine damage. The temporal behavior of this sensor is that it will show an increase in temperature over time as the engine continues to run with a clogged filter or low

KeyboardInterrupt: 

In [38]:
sensor_json_res = {'results': []}
for component in results:
    for i, sensor_desc in enumerate(results[component]):
        try:
            logging.debug(f'{component}, {i}')
            prompt = f'{sensor_desc}\nGenerate json object with two key: sensor name, temporal behavior, and description for a given failure event.'
            json_res = generate(prompt)
            start = json_res.find('{')
            end = json_res.rfind('}')
            json_res = json.loads(json_res[start:end+1])
            sensor_json_res['results'].append(json_res)
            logging.debug(json_res)
        except:
            logging.debug(f'error {component}, {i}')
        with open(f'{asset_name}_json_conv.json', 'w') as f:
            f.write(json.dumps(sensor_json_res))

error Engine
